In [1]:
from pathlib import Path
from itertools import product
import polars as pl
from sklearn.model_selection import StratifiedKFold
import pandera.polars as pa
import colorlog
logger = colorlog.getLogger()

In [2]:
def get_data(
    et_bin_left: pl.lit,
    et_bin_right: pl.lit,
    eta_bin_left: pl.lit,
    eta_bin_right: pl.lit,
    et_col: pl.Expr,
    eta_col: pl.Expr,
    el_medium_col: pl.Expr,
    el_vloose_col: pl.Expr,
    target_col: pl.Expr,
    table_path: Path,
) -> pl.LazyFrame:
    et_col = pl.when(et_col < 0).then(pl.lit(0, dtype=pl.dtype_of(et_col))).otherwise(et_col)
    eta_col = eta_col.abs()
    label = pl \
        .when(el_medium_col & (target_col == 1)).then(pl.lit(True, dtype=pl.Boolean)) \
        .when(~el_vloose_col & (target_col == 0)).then(pl.lit(False, dtype=pl.Boolean)) \
        .otherwise(pl.lit(None)).alias('label')

    df = pl.scan_parquet(str(table_path))
    df = df.filter(
        et_col.is_between(et_bin_left, et_bin_right, closed='left') &
        eta_col.is_between(eta_bin_left, eta_bin_right, closed='left') &
        label.is_not_null()
    ).select(
        pl.col('id'), label
    )
    return df

In [3]:
dataset_dir = Path('../tests/data/test_dataset')
electron_ringer_path = dataset_dir / 'electron_ringer.parquet'
standard_binning_kfold_path = dataset_dir / 'standard_binning_kfold.parquet'

In [4]:
et_bins = [
    pl.lit(15*1000, dtype=pl.Float32),
    pl.lit(20*1000, dtype=pl.Float32),
    pl.lit(30*1000, dtype=pl.Float32),
    pl.lit(40*1000, dtype=pl.Float32),
    pl.lit(50*1000, dtype=pl.Float32),
    pl.lit(float('inf'), dtype=pl.Float32)]
et_col = pl.col('TrigEMClusterContainer.et')
eta_bins = [
    pl.lit(0, dtype=pl.Float32),
    pl.lit(0.8, dtype=pl.Float32),
    pl.lit(1.37, dtype=pl.Float32),
    pl.lit(1.52, dtype=pl.Float32),
    pl.lit(2.500001, dtype=pl.Float32)
]
eta_col = pl.col('TrigEMClusterContainer.eta')
el_medium_col = pl.col('OfflineElectronTriggerContainer.lhmedium')
el_vloose_col = pl.col('OfflineElectronTriggerContainer.lhvloose')
target_col = pl.col('target')
splitter = StratifiedKFold(n_splits=5, shuffle=True)

In [5]:
all_dfs = []
bins_iterator = product(zip(et_bins[:-1], et_bins[1:]), zip(eta_bins[:-1], eta_bins[1:]))
for i, ((et_bin_left, et_bin_right), (eta_bin_left, eta_bin_right)) in enumerate(bins_iterator):
    logger.info(f'{i} - Processing bin: et [{et_bin_left}, {et_bin_right}), eta [{eta_bin_left}, {eta_bin_right})')
    df = get_data(
        et_bin_left=et_bin_left,
        et_bin_right=et_bin_right,
        eta_bin_left=eta_bin_left,
        eta_bin_right=eta_bin_right,
        et_col=et_col,
        eta_col=eta_col,
        el_medium_col=el_medium_col,
        el_vloose_col=el_vloose_col,
        target_col=target_col,
        table_path=electron_ringer_path,
    ).collect()
    id_values = df.get_column('id').to_numpy().reshape(-1, 1)
    label_values = df.get_column('label').to_numpy().reshape(-1, 1)
    fold_ids = []
    for fold, (_, val_idx) in enumerate(splitter.split(id_values, label_values)):
        fold_ids.append(id_values[val_idx].flatten())
    kfold_col = pl.when(pl.col('id').is_in(fold_ids[0])).then(pl.lit(0, dtype=pl.UInt8))
    for fold, fold_id in enumerate(fold_ids[1:], start=1):
        kfold_col = kfold_col.when(pl.col('id').is_in(fold_id)).then(pl.lit(fold, dtype=pl.UInt8))
    kfold_col = kfold_col.otherwise(pl.lit(None))
    df = df.with_columns(kfold_col.alias('kfold'))
    all_dfs.append(df)

In [6]:
final_df = pl.concat(all_dfs)
print(final_df.schema)
final_df

Schema({'id': UInt64, 'label': Boolean, 'kfold': UInt8})


id,label,kfold
u64,bool,u8
45,true,2
249,true,0
295,true,2
445,true,3
478,false,1
…,…,…
199951,true,4
199956,true,1
199960,false,4


# Validate

In [7]:
kfold_dataframe_schema = pa.DataFrameSchema({
    'id': pa.Column(
        pl.UInt64,
        unique=True,
        checks=[
            pa.Check.ge(0),
        ]),
    'label': pa.Column(
        pl.Boolean,
        nullable=True),
    'kfold': pa.Column(
        pl.UInt8,
        nullable=True,
        checks=[
            pa.Check.ge(0),
            pa.Check.le(4),
        ])
})

In [8]:
final_df = kfold_dataframe_schema.validate(final_df)
final_df

id,label,kfold
u64,bool,u8
45,true,2
249,true,0
295,true,2
445,true,3
478,false,1
…,…,…
199951,true,4
199956,true,1
199960,false,4


In [9]:
final_df.write_parquet(pl.PartitionBy(
    str(standard_binning_kfold_path),
    max_rows_per_file=100_000,
), compression='snappy')